# SCT continuation 
Markers, annotations, annotated UMAPs, and tissue distribution heatmaps for SCT data d15/d20/d25 at r0.3 and r0.5.

In [1]:
library(Seurat)
library(ggplot2)
library(pheatmap)

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built under R 4.4.0 but the current version is
4.4.3; it is recomended that you reinstall ‘SeuratObject’ as the ABI
for R may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




In [2]:
vis_dir <- file.path("..", "vis", "SRT_annotation_vis")
data_dir <- file.path("..", "data", "SRT_annotation_data")
sweep_dir <- file.path("..", "data", "sct_sweep_results")

dir.create(vis_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(data_dir, showWarnings = FALSE, recursive = TRUE)

files <- c(
  "seur_sct_d15_r0.3.rds",
  "seur_sct_d15_r0.5.rds",
  "seur_sct_d20_r0.3.rds",
  "seur_sct_d20_r0.5.rds",
  "seur_sct_d25_r0.3.rds",
  "seur_sct_d25_r0.5.rds"
 )

files <- file.path(sweep_dir, files)

In [3]:
# marker-driven annotation settings
top_n_markers <- 10
marker_label_prefix <- "Cluster"

In [ ]:
for (f in files) {
  obj <- readRDS(f)
  label <- gsub("seur_sct_|\\.rds$", "", basename(f))

  # ensure metadata rownames match cell names
  if (!identical(rownames(obj@meta.data), colnames(obj))) {
    rownames(obj@meta.data) <- colnames(obj)
  }

  markers_path <- file.path(data_dir, paste0("markers_", label, ".csv"))
  top_markers_path <- file.path(data_dir, paste0("markers_top", top_n_markers, "_", label, ".csv"))

  # markers (reuse existing if available)
  if (file.exists(markers_path)) {
    markers <- read.csv(markers_path)
  } else {
    markers <- FindAllMarkers(
      obj,
      group.by = "seurat_clusters",
      assay = "SCT",
      logfc.threshold = 1
    )
    write.csv(markers, markers_path, row.names = FALSE)
  }

  markers <- markers[order(markers$cluster, -markers$avg_log2FC), ]
  if (file.exists(top_markers_path)) {
    top_markers <- read.csv(top_markers_path)
  } else {
    top_markers <- do.call(rbind, lapply(split(markers, markers$cluster), head, top_n_markers))
    write.csv(top_markers, top_markers_path, row.names = FALSE)
  }

  # marker-derived annotations (label by top gene only)
  top_gene <- tapply(markers$gene, markers$cluster, function(x) x[1])
  cluster_annotations <- setNames(top_gene, names(top_gene))
  celltype_vec <- cluster_annotations[as.character(obj$seurat_clusters)]
  names(celltype_vec) <- colnames(obj)
  obj <- AddMetaData(obj, metadata = celltype_vec, col.name = "celltype")

  # annotated UMAP (legend shows top gene; no cluster labels on plot)
  umap_path <- file.path(vis_dir, paste0("umap_celltype_", label, ".png"))
  if (!file.exists(umap_path)) {
    p_umap <- DimPlot(obj, group.by = "celltype", label = FALSE) +
      ggtitle(paste("SCT", label, "celltype"))
    ggsave(umap_path, p_umap, width = 8, height = 6, dpi = 200)
  }

  # tissue distribution tables + heatmap (tissue on bottom, top genes on right)
  tissue_col <- if ("TissueType" %in% colnames(obj@meta.data)) {
    "TissueType"
  } else if ("Sample" %in% colnames(obj@meta.data)) {
    "Sample"
  } else {
    "orig.ident"
  }

  tab <- table(obj$celltype, obj@meta.data[[tissue_col]])
  tab_pct <- round(100 * prop.table(tab, margin = 2), 1)

  counts_path <- file.path(data_dir, paste0("tissue_cluster_counts_", label, ".csv"))
  pct_path <- file.path(data_dir, paste0("tissue_cluster_pct_", label, ".csv"))
  if (!file.exists(counts_path)) {
    write.csv(tab, counts_path)
  }
  if (!file.exists(pct_path)) {
    write.csv(tab_pct, pct_path)
  }

  heatmap_path <- file.path(vis_dir, paste0("tissue_distribution_", label, ".png"))
  if (!file.exists(heatmap_path)) {
    pheatmap(tab_pct, cluster_rows = FALSE, cluster_cols = FALSE,
             filename = heatmap_path,
             width = 6, height = 6)
  }
}

Calculating cluster 0

Calculating cluster 1

Calculating cluster 2

Calculating cluster 3

Calculating cluster 4

Calculating cluster 5

